In [2]:
%pip show cmd2

Name: cmd2
Version: 2.4.3
Summary: cmd2 - quickly build feature-rich and user-friendly interactive command line applications in Python
Home-page: https://github.com/python-cmd2/cmd2
Author: Catherine Devlin
Author-email: catherine.devlin@gmail.com
License: MIT
Location: /headless/.local/lib/python3.12/site-packages
Requires: attrs, pyperclip, wcwidth
Required-by: PrettyPrintTree
Note: you may need to restart the kernel to use updated packages.


In [1]:
import sys
print(sys.executable)
print(sys.version)

from glayout.flow.pdk.mappedpdk import MappedPDK
from glayout.flow.pdk.sky130_mapped import sky130_mapped_pdk as sky130
from glayout.flow.pdk.gf180_mapped import gf180_mapped_pdk as gf180

from gdsfactory import Component
from gdsfactory.components import text_freetype, rectangle

from cmd2.ansi import style_aware_wcswidth
from PrettyPrint import PrettyPrintTree

from glayout.flow.primitives.fet import nmos, pmos
from glayout.flow.primitives.via_gen import via_stack
from glayout.flow.pdk.util.port_utils import rename_ports_by_orientation
from glayout.flow.primitives.guardring import tapring

from glayout.flow.routing.straight_route import straight_route
from glayout.flow.routing.c_route import c_route
from glayout.flow.routing.L_route import L_route


from glayout.flow.primitives.mimcap import mimcap  

/foss/designs/layout/glayout_env/bin/python
3.10.16 (main, Mar 11 2025, 17:27:26) [Clang 20.1.0 ]


ModuleNotFoundError: No module named 'glayout.flow'

In [2]:
import os

run_drc_py = (
    "/foss/pdks/ciel/gf180mcu/"
    "versions/7b70722e33c03fcb5dabcf4d479fb0822d9251c9/"
    "gf180mcuD/libs.tech/klayout/tech/drc/run_drc.py"
)

print("Existe:", os.path.isfile(run_drc_py))
print(run_drc_py)

with open(run_drc_py, "r") as f:
    lines = f.readlines()

for i in range(min(160, len(lines))):
    print(f"{i+1:03d}: {lines[i].rstrip()}")

Existe: True
/foss/pdks/ciel/gf180mcu/versions/7b70722e33c03fcb5dabcf4d479fb0822d9251c9/gf180mcuD/libs.tech/klayout/tech/drc/run_drc.py
001: ################################################################################################
002: # Copyright 2023 GlobalFoundries PDK Authors
003: #
004: # Licensed under the Apache License, Version 2.0 (the "License");
005: # you may not use this file except in compliance with the License.
006: # You may obtain a copy of the License at
007: #
008: #     https://www.apache.org/licenses/LICENSE-2.0
009: #
010: # Unless required by applicable law or agreed to in writing, software
011: # distributed under the License is distributed on an "AS IS" BASIS,
012: # WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
013: # See the License for the specific language governing permissions and
014: # limitations under the License.
015: ################################################################################################
016

In [4]:
from glayout.flow.pdk.util.comp_utils import (
    evaluate_bbox,
    prec_center,
    prec_ref_center,
    align_comp_to_port,
)

from glayout.flow.pdk.util.port_utils import (
    add_ports_perimeter,
    print_ports,
)

from glayout.flow.pdk.util.snap_to_grid import (
    component_snap_to_grid,
)

from glayout.flow.spice.netlist import Netlist

In [5]:
from pathlib import Path

# 1. Definimos la carpeta exacta de tu proyecto OpenFASOC en el entorno Linux
carpeta_proyecto = Path("/foss/designs/layout")

# 2. Definimos la ruta completa del archivo
#ruta_gds = carpeta_proyecto / "test_inverter.gds"



In [23]:
nmos_kwargs = {
    "with_tie": True,
    "with_dnwell": True,
    "sd_route_topmet": "met2",
    "gate_route_topmet": "met2",
    "sd_route_left": True,
    "rmult": None,
    "gate_rmult": 1,
    "interfinger_rmult": 1,
    "substrate_tap_layers": ("met2","met1"),
    "dummy_routes": True
}

pmos_kwargs = {
    "with_tie": True,
    "dnwell": False,
    "sd_route_topmet": "met2",
    "gate_route_topmet": "met2",
    "sd_route_left": True,
    "rmult": None,
    "gate_rmult": 1,
    "interfinger_rmult": 1,
    "substrate_tap_layers": ("met2","met1"),
    "dummy_routes": True
}

mimcap_kwargs = {
    "option": "B",
    "with_extension": True,
    "extension_direction": "S",
}     

In [7]:
import os
import glob
import subprocess
import xml.etree.ElementTree as ET
from glayout.flow.pdk.gf180_mapped import gf180_mapped_pdk as gf180
import time


def imprimir_resumen_drc(ruta_reporte):
    """Parsea el archivo .lyrdb de KLayout e imprime un resumen limpio de los errores."""
    try:
        tree = ET.parse(ruta_reporte)
        root = tree.getroot()

        # 1. Mapear nombres de categoría a sus descripciones
        descripciones = {}
        for cat in root.findall(".//category"):
            nombre = cat.find("name")
            desc = cat.find("description")
            if nombre is not None and nombre.text:
                descripciones[nombre.text.strip("'\"")] = (
                    desc.text if desc is not None else ""
                )

        # 2. Contar violaciones reales por categoría
        conteo_errores = {}
        total_errores = 0

        for item in root.findall(".//item"):
            cat_item = item.find("category")

            if cat_item is not None and cat_item.text:
                cat_nombre = cat_item.text.strip("'\"")

                conteo_errores[cat_nombre] = (
                    conteo_errores.get(cat_nombre, 0) + 1
                )

                total_errores += 1

        # 3. Mostrar resumen en consola
        print("\n" + "=" * 70)
        print(
            f"?? RESUMEN DE ERRORES DRC | "
            f"Total de violaciones: {total_errores}"
        )
        print("=" * 70)

        if total_errores == 0:
            print("  ?? ¡Felicidades! No se encontraron violaciones DRC.")
        else:
            for cat, cantidad in conteo_errores.items():
                desc = descripciones.get(cat, "Sin descripción")

                print(
                    f" ? [{cat}] "
                    f"({cantidad} error{'es' if cantidad > 1 else ''}): "
                    f"{desc}"
                )

        print("=" * 70 + "\n")

    except Exception as e:
        print(
            f"?? No se pudo procesar el resumen XML del reporte: {e}"
        )


def run_drc_v2(ruta_gds):

    ruta_gds_str = str(ruta_gds)

    if not os.path.exists(ruta_gds_str):
        print(f"? Error: No existe el archivo {ruta_gds_str}")
        return

    print(f"?? Ejecutando DRC para: {ruta_gds_str} ...")

    # ==========================================================
    # CAMBIO: usar el DRC OFICIAL DEL GF180MCU
    # ==========================================================

    gf180_root = (
        "/foss/pdks/ciel/gf180mcu/versions/"
        "7b70722e33c03fcb5dabcf4d479fb0822d9251c9/gf180mcuD"
    )

    run_drc_py = os.path.join(
        gf180_root,
        "libs.tech",
        "klayout",
        "tech",
        "drc",
        "run_drc.py"
    )

    dir_gds = os.path.dirname(os.path.abspath(ruta_gds_str))

    comando = [
        "python3",
        run_drc_py,
        f"--path={ruta_gds_str}",
        "--variant=B",
        f"--run_dir={dir_gds}",
        "--verbose"
    ]

    resultado = subprocess.run(
        comando,
        capture_output=True,
        text=True
    )

    es_clean = resultado.returncode == 0

    print(
        f"¿Diseño libre de errores (DRC Clean)?: {es_clean}"
    )

    # Mostrar salida del DRC oficial si existe
    if resultado.stdout:
        print("\n----- DRC STDOUT -----")
        print(resultado.stdout)

    if resultado.stderr:
        print("\n----- DRC STDERR -----")
        print(resultado.stderr)

    # Carpetas donde buscaremos el reporte
    dir_actual = os.getcwd()
    dir_foss = "/foss/designs/layout"

    carpetas_busqueda = [
        dir_foss,
        dir_gds,
        dir_actual,
        "/tmp"
    ]

    # También incluimos el directorio donde el GF180
    # puede generar los reportes internamente.
    dir_run_drc = os.path.join(
        gf180_root,
        "libs.tech",
        "klayout",
        "tech",
        "macros",
        "run_drc_main"
    )

    carpetas_busqueda.append(dir_run_drc)

    # Buscamos archivos .lyrdb, .ylrdb o que contengan '_drcreport'
    archivos_encontrados = []

    for carpeta in carpetas_busqueda:

        if not os.path.isdir(carpeta):
            continue

        for patron in [
            "*_drcreport*",
            "*.lyrdb",
            "*.ylrdb"
        ]:
            ruta_busqueda = os.path.join(
                carpeta,
                patron
            )

            archivos_encontrados.extend(
                glob.glob(ruta_busqueda)
            )

    # Filtramos para asegurarnos de que sean archivos válidos
    archivos_validos = [
        f
        for f in set(archivos_encontrados)
        if os.path.isfile(f)
    ]

    if archivos_validos:

        reporte_reciente = max(
            archivos_validos,
            key=os.path.getmtime
        )

        print(
            f"?? Reporte de DRC detectado en: "
            f"{reporte_reciente}"
        )

        # --- IMPRIMIR RESUMEN SIMPLIFICADO ---
        imprimir_resumen_drc(reporte_reciente)

        os.system("pkill -f klayout")
        time.sleep(1)
        
        print("?? Abriendo KLayout...")

        subprocess.Popen([
            "klayout",
            "-e",
            ruta_gds_str,
            "-m",
            reporte_reciente
        ])

    else:

        print(
            "?? No se encontró ningún reporte de DRC "
            "en las rutas esperadas."
        )

In [8]:
import os
import glob
import subprocess
import time
from pathlib import Path

def verificar_drc_manual(nombre_archivo="tg_layout.gds"):
    ruta_gds = os.path.abspath(nombre_archivo)
    dir_gds = os.path.dirname(ruta_gds)

    if not os.path.exists(ruta_gds):
        print(f"? Error: No existe el archivo {ruta_gds}")
        print("Asegúrate de haberlo guardado desde KLayout.")
        return

    print(f"?? Ejecutando DRC para: {ruta_gds} ...")

    # Rutas del PDK GF180
    gf180_root = "/foss/pdks/ciel/gf180mcu/versions/7b70722e33c03fcb5dabcf4d479fb0822d9251c9/gf180mcuD"
    run_drc_py = os.path.join(gf180_root, "libs.tech", "klayout", "tech", "drc", "run_drc.py")

    # Comando de ejecución
    comando = [
        "python3",
        run_drc_py,
        f"--path={ruta_gds}",
        "--variant=B",
        f"--run_dir={dir_gds}"
    ]

    # Ejecutar en segundo plano esperando el resultado
    resultado = subprocess.run(comando, capture_output=True, text=True)
    
    if resultado.returncode == 0:
         print("? DRC Terminado con éxito (el script corrió bien).")
    else:
         print("?? DRC finalizó con errores de ejecución (Revisar logs).")

    # Buscar el archivo .lyrdb recién generado
    archivos_encontrados = glob.glob(os.path.join(dir_gds, "*.lyrdb")) + \
                           glob.glob(os.path.join(dir_gds, "*_drcreport*"))
                           
    archivos_validos = [f for f in set(archivos_encontrados) if os.path.isfile(f)]

    if archivos_validos:
        # Tomar el más reciente
        reporte_reciente = max(archivos_validos, key=os.path.getmtime)
        print(f"?? Reporte de DRC detectado en: {reporte_reciente}")

        # Cerrar KLayout viejo para evitar el problema del sufijo $2
        print("?? Limpiando memoria de KLayout...")
        os.system("pkill -f klayout")
        time.sleep(1) # Pausa breve para asegurar el cierre

        # Abrir KLayout nuevo con el diseño editable y los marcadores listos
        print("?? Abriendo KLayout...")
        subprocess.Popen([
            "klayout",
            "-e",          # Modo editable
            ruta_gds,      # Tu archivo modificado
            "-m",          # Cargar marcadores
            reporte_reciente
        ])
    else:
        print("? No se generó ningún archivo .lyrdb de marcadores.")


In [25]:
from pathlib import Path
import time
import gdsfactory as gf
from gdsfactory.component import Component
from glayout.flow.pdk.mappedpdk import MappedPDK
from glayout.flow.primitives.fet import nmos, pmos
from glayout.flow.routing.c_route import c_route
from glayout.flow.pdk.gf180_mapped import gf180_mapped_pdk as gf180

def transmission_gate(
    pdk: MappedPDK, 
    Wp: float = 4.5, 
    Wn: float = 1.3, 
    Lp: float = 0.36, 
    Ln: float = 0.36, 
    fing: int = 1, 
    mult: int = 1
) -> Component:
    """
    Transmission Gate Ultracompacta DRC Clean (GF180MCU).
    """
    nombre_seguro = "switch"
    tg = Component(nombre_seguro)

    clean_params = {
        "sd_route_topmet": "met3",
        "gate_route_topmet": "met3",
        "sd_route_left": False,
        "dummy_routes": True,
        "tie_layers": ("met1", "met1")
    }

    # =====================================================================
    # 1. CELDAS CON TIES ACTIVADOS (Sin Tap Rings)
    # =====================================================================
    pfet_cell = pmos(
        pdk, width=Wp, length=Lp, fingers=fing, multipliers=mult,
        with_substrate_tap=False, # <-- Tap ring desactivado
        with_dummy=(True, True), 
        with_tie=True,            # <-- Solo Tie activado
        **clean_params
    )

    nfet_cell = nmos(
        pdk, width=Wn, length=Ln, fingers=fing, multipliers=mult,
        with_substrate_tap=False, # <-- Tap ring desactivado
        with_dnwell=False, 
        with_dummy=(True,True), 
        with_tie=True,            # <-- Solo Tie activado
        **clean_params
    )

    pfet_ref = tg << pfet_cell
    nfet_ref = tg << nfet_cell

    # =====================================================================
    # 2. FLOORPLAN Y ESPACIAMIENTO SEGURO
    # =====================================================================
    # Distancia segura entre pozos (~2.0um para evitar problemas de espaciado NW.4)
    well_isolation = 2.0

    pfet_ref.move((0, 0))
    pfet_ref.mirror_y()

    nfet_ref.x = pfet_ref.x
    nfet_ref.ymax = pfet_ref.ymin - well_isolation

    # =====================================================================
    # 3. REFUERZO DE POZOS POR FUERZA BRUTA (Resuelve DF.4c_LV)
    # =====================================================================
    # Añadimos 0.6um extra en todas las direcciones para superar los 0.43um requeridos
    margin = 0.6
    
    nwell_poly = [
        (pfet_ref.xmin - margin, pfet_ref.ymin - margin),
        (pfet_ref.xmax + margin, pfet_ref.ymin - margin),
        (pfet_ref.xmax + margin, pfet_ref.ymax + margin),
        (pfet_ref.xmin - margin, pfet_ref.ymax + margin)
    ]
    tg.add_polygon(nwell_poly, layer=pdk.get_glayer("nwell"))

    # Hacemos lo mismo para el PWELL del NMOS por simetría y seguridad
    pwell_poly = [
        (nfet_ref.xmin - margin, nfet_ref.ymin - margin),
        (nfet_ref.xmax + margin, nfet_ref.ymin - margin),
        (nfet_ref.xmax + margin, nfet_ref.ymax + margin),
        (nfet_ref.xmin - margin, nfet_ref.ymax + margin)
    ]
    try:
        tg.add_polygon(pwell_poly, layer=pdk.get_glayer("pwell"))
    except Exception:
        pass # Por si la versión de GLAYOUT no tiene pwell explícito mapeado

    # =====================================================================
    # 4. ENRUTAMIENTO EN METAL 3
    # =====================================================================
    p_src_port = pfet_ref.ports.get("multiplier_0_source_W", pfet_ref.ports.get("source_W"))
    n_src_port = nfet_ref.ports.get("multiplier_0_source_W", nfet_ref.ports.get("source_W"))
    
    p_drn_port = pfet_ref.ports.get("multiplier_0_drain_E", pfet_ref.ports.get("drain_E"))
    n_drn_port = nfet_ref.ports.get("multiplier_0_drain_E", nfet_ref.ports.get("drain_E"))

    in_route = c_route(pdk, p_src_port, n_src_port, e1glayer="met3", e2glayer="met3", cglayer="met3", extension=1.8)
    tg << in_route

    out_route = c_route(pdk, p_drn_port, n_drn_port, e1glayer="met3", e2glayer="met3", cglayer="met3", extension=1.8)
    tg << out_route

    return tg


# =====================================================================
# Script de prueba / generación
# =====================================================================
gf.clear_cache()
tg_layout = transmission_gate(gf180, Wp=4.5, Wn=1.3, Lp=0.36, Ln=0.36, mult=1, fing=1)
ruta_gds_tg = "switch.gds"
tg_layout.write_gds(ruta_gds_tg)

print(f"? Transmission Gate generada y guardada en: {ruta_gds_tg}")
run_drc_v2(ruta_gds_tg)

/tmp/ipykernel_42142/2091829499.py:118: UserWarning: Unnamed cells, 2 in 'switch$14'
  tg_layout.write_gds(ruta_gds_tg)
2026-08-17 04:26:39.936 | INFO     | gdsfactory.component:_write_library:1851 - Wrote to 'switch.gds'


? Transmission Gate generada y guardada en: switch.gds
?? Ejecutando DRC para: switch.gds ...
¿Diseño libre de errores (DRC Clean)?: False

----- DRC STDOUT -----
2026-08-17 04:26:43 +0200: Memory Usage (448016K) : Starting running GF180MCU Klayout DRC runset on /foss/designs/layout/switch.gds
2026-08-17 04:26:43 +0200: Memory Usage (448016K) : Ruby Version for klayout: 3.2.3


----- DRC STDERR -----
/foss/pdks/ciel/gf180mcu/versions/7b70722e33c03fcb5dabcf4d479fb0822d9251c9/gf180mcuD/libs.tech/klayout/tech/drc/run_drc.py:729: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now_str = datetime.utcnow().strftime("drc_run_%Y_%m_%d_%H_%M_%S")
17-Aug-2026 04:26:41 | INFO    | Your Klayout version is: KLayout 0.30.8
17-Aug-2026 04:26:41 | INFO    | ## Generating template with for the following rule tables: ['dummy_metal4.drc', 'dummy_excl

ERROR: In /foss/pdks/ciel/gf180mcu/versions/7b70722e33c03fcb5dabcf4d479fb0822d9251c9/gf180mcuD/libs.tech/klayout/tech/macros/gf180mcu_lvs.lylvs: Can't find a schematic counterpart for the top cell switch$14 - use 'same_circuit' to establish a correspondence
ERROR: :/built-in-macros/_lvs_netter.rb:302: RuntimeError: Can't find a schematic counterpart for the top cell switch$14 - use 'same_circuit' to establish a correspondence in Executable::execute
  :/built-in-macros/_lvs_netter.rb:302:in `align'
  (eval):2:in `align'
  /foss/pdks/ciel/gf180mcu/versions/7b70722e33c03fcb5dabcf4d479fb0822d9251c9/gf180mcuD/libs.tech/klayout/tech/macros/../lvs/gf180mcu.lvs:358:in `execute'
  :/built-in-macros/lvs_interpreters.lym:31:in `instance_eval'
  :/built-in-macros/lvs_interpreters.lym:31:in `execute'


2026-08-17 04:42:34 +0200: Memory Usage (3418716K) : Starting GF180 LVS comparison section
2026-08-17 04:42:44 +0200: Memory Usage (3427380K) : Starting running GF180MCU Klayout LVS runset on 
2026-08-17 04:42:44 +0200: Memory Usage (3427380K) : Ruby Version for klayout: 3.2.3
2026-08-17 04:42:44 +0200: Memory Usage (3427380K) : Loading database to memory is complete.
2026-08-17 04:42:44 +0200: Memory Usage (3427380K) : GF180MCU Klayout LVS runset output at default location: switch.lvsdb
2026-08-17 04:42:44 +0200: Memory Usage (3427380K) : Evaluate switches.
2026-08-17 04:42:44 +0200: Memory Usage (3427380K) : Substrate name used: gf180mcu_gnd
2026-08-17 04:42:44 +0200: Memory Usage (3427380K) : Extracted netlist with net names: true
2026-08-17 04:42:44 +0200: Memory Usage (3427380K) : Extracted netlist with comments in details: false
2026-08-17 04:42:44 +0200: Memory Usage (3427380K) : GF180MCU Klayout LVS extracted netlist file at: switch_extracted.cir
2026-08-17 04:42:44 +0200: Memo


(klayout:98079): GVFS-RemoteVolumeMonitor-WARNING **: 05:42:39.885: remote volume monitor with dbus name org.gtk.vfs.UDisks2VolumeMonitor is not supported


(22, 0)
## gf180mcu PDK Pcells loaded.
['/foss/pdks/ciel/gf180mcu/versions/7b70722e33c03fcb5dabcf4d479fb0822d9251c9/gf180mcuD/libs.tech/klayout/tech/pymacros', '/foss/tools/klayout/pymod', '/foss/tools/klayout_gdsfactory9/lib/python3.12/site-packages', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '/headless/.local/lib/python3.12/site-packages', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/headless/.klayout/python', '/headless/.klayout/salt/KLayoutPluginUtils/python', '/headless/.klayout/salt/klive/python']
klive 0.4.1 is running
2026-08-17 05:36:12 +0200: Memory Usage (3583572K) : xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx


In [60]:
verificar_drc_manual("switch.gds")

?? Ejecutando DRC para: /foss/designs/layout/switch.gds ...
?? DRC finalizó con errores de ejecución (Revisar logs).
?? Reporte de DRC detectado en: /foss/designs/layout/switch_main.lyrdb
?? Limpiando memoria de KLayout...
?? Abriendo KLayout...
2026-08-21 06:30:30 +0200: Memory Usage (3180936K) : Starting running GF180MCU Klayout LVS runset on 
2026-08-21 06:30:30 +0200: Memory Usage (3180936K) : Ruby Version for klayout: 3.2.3
2026-08-21 06:30:30 +0200: Memory Usage (3180936K) : Loading database to memory is complete.
2026-08-21 06:30:30 +0200: Memory Usage (3180936K) : GF180MCU Klayout LVS runset output at default location: switch.lvsdb
2026-08-21 06:30:30 +0200: Memory Usage (3180936K) : Evaluate switches.
2026-08-21 06:30:30 +0200: Memory Usage (3180936K) : Substrate name used: gf180mcu_gnd
2026-08-21 06:30:30 +0200: Memory Usage (3180936K) : Extracted netlist with net names: true
2026-08-21 06:30:30 +0200: Memory Usage (3180936K) : Extracted netlist with comments in details: fals

In [32]:
from pathlib import Path
import time
import gdsfactory as gf
from gdsfactory.component import Component
from glayout.flow.pdk.mappedpdk import MappedPDK
from glayout.flow.primitives.fet import nmos, pmos
from glayout.flow.primitives.guardring import tapring
from glayout.flow.routing.straight_route import straight_route
from glayout.flow.routing.c_route import c_route
from glayout.flow.pdk.gf180_mapped import gf180_mapped_pdk as gf180

def inverter_abutted(pdk: MappedPDK, Wp=6.0, Wn=3.0, Lp=1.0, Ln=1.0):
    nombre_seguro = f"inv_layout_{int(time.time())}"
    inv = Component(nombre_seguro)

    # =========================================================================
    # 1. PARAMETROS LIMPIOS: SIN TAPS INDIVIDUALES PARA PERMITIR ABUTMENT
    # =========================================================================
    clean_params = {
        "sd_route_topmet": "met3",   # Source y Drain escalan máximo a met2
        "gate_route_topmet": "met1", # Gates se quedan en met1
        "sd_route_left": False,
        "dummy_routes": True,
        "with_tie": False,           # <- APAGADO para no superponer abutment
        "with_substrate_tap": False, # <- APAGADO para no superponer abutment
        "tie_layers": ("met1", "met1")
    }

    # =========================================================================
    # 2. PMOS ABUTTED PAIR
    # =========================================================================
    pfet1 = pmos(pdk, width=Wp, length=Lp, multipliers=1, with_dummy=(True, False), dnwell=False, **clean_params)
    pfet2 = pmos(pdk, width=Wp, length=Lp, multipliers=1, with_dummy=(True, False), dnwell=False, **clean_params)

    p1 = inv << pfet1
    p2 = inv << pfet2

    p2.mirror_x()
    p2.x += (p1.ports["drain_E"].x - p2.ports["drain_E"].x) - 0.5
    p2.ymin = p1.ymin

    # =========================================================================
    # 3. NMOS ABUTTED PAIR
    # =========================================================================
    nfet1 = nmos(pdk, width=Wn, length=Ln, multipliers=1, with_dummy=(True, False), with_dnwell=False, **clean_params)
    nfet2 = nmos(pdk, width=Wn, length=Ln, multipliers=1, with_dummy=(True, False), with_dnwell=False, **clean_params)

    n1 = inv << nfet1
    n2 = inv << nfet2

    n2.mirror_x()
    n1.mirror_y()
    n2.mirror_y()

    n1.x = p1.x
    n2.x += (n1.ports["drain_E"].x - n2.ports["drain_E"].x) - 0.5

    spacing_y = 9.0
    n1.ymax = p1.ymin - spacing_y
    n2.ymax = p1.ymin - spacing_y

    # =========================================================================
    # 4. TAP RINGS EXTERNOS (LA SOLUCIÓN PARA BLOQUES ABUTTED)
    # =========================================================================
    margin = 1.8

    # PMOS TapRing (NWell Tap)
    p_xmin, p_xmax = min(p1.xmin, p2.xmin), max(p1.xmax, p2.xmax)
    p_ymin, p_ymax = min(p1.ymin, p2.ymin), max(p1.ymax, p2.ymax)
    
    p_tap = tapring(
        pdk, 
        enclosed_rectangle=((p_xmax - p_xmin) + 2 * margin, (p_ymax - p_ymin) + 2 * margin), 
        sdlayer="n+s/d",         # Dopaje inyectado directamente en el anillo
        vertical_glayer='met2'
    )
    p_tap_ref = inv << p_tap
    p_tap_ref.x = (p_xmin + p_xmax) / 2.0
    p_tap_ref.y = (p_ymin + p_ymax) / 2.0

    # NMOS TapRing (PSub Tap)
    n_xmin, n_xmax = min(n1.xmin, n2.xmin), max(n1.xmax, n2.xmax)
    n_ymin, n_ymax = min(n1.ymin, n2.ymin), max(n1.ymax, n2.ymax)
    
    n_tap = tapring(
        pdk, 
        enclosed_rectangle=((n_xmax - n_xmin) + 2 * margin, (n_ymax - n_ymin) + 2 * margin), 
        sdlayer="p+s/d",         # Dopaje inyectado directamente en el anillo
        vertical_glayer='met2'
    )
    n_tap_ref = inv << n_tap
    n_tap_ref.x = (n_xmin + n_xmax) / 2.0
    n_tap_ref.y = (n_ymin + n_ymax) / 2.0

    # =========================================================================
    # 5. NWELL GIGANTE PARA PMOS (Resuelve DF.4c_LV y DF.13)
    # =========================================================================
    nw_margin = 0.6
    inv.add_polygon(
        [
            (p_tap_ref.xmin - nw_margin, p_tap_ref.ymin - nw_margin),
            (p_tap_ref.xmax + nw_margin, p_tap_ref.ymin - nw_margin),
            (p_tap_ref.xmax + nw_margin, p_tap_ref.ymax + nw_margin),
            (p_tap_ref.xmin - nw_margin, p_tap_ref.ymax + nw_margin),
        ],
        layer=pdk.get_glayer("nwell")
    )

    # =========================================================================
    # 6. ENRUTAMIENTO (Met1 para Gates, Met2 para Drains)
    # =========================================================================
    p1_g = p1.ports["gate_S"]
    p2_g = p2.ports["gate_S"]
    n1_g = n1.ports["gate_S"]
    n2_g = n2.ports["gate_S"]

    inv << straight_route(pdk, p1_g, n1_g, glayer1="met3")
    inv << straight_route(pdk, p2_g, n2_g, glayer1="met3")
    inv << straight_route(pdk, p1_g, p2_g, glayer1="met1")

    inv << c_route(
        pdk, 
        p1.ports["drain_E"], 
        n1.ports["drain_E"], 
        e1glayer="met3", 
        e2glayer="met3", 
        cglayer="met2",
        extension=8.2
    )

    # =========================================================================
    # 7. LIMPIEZA
    # =========================================================================
    inv.remove_layers(layers=[(204, 0)])

    return inv

    
    
ruta_gds= carpeta_proyecto 

ruta_gds_inverter = carpeta_proyecto / "inverter.gds"
    
# Visualization
inverter_layout = inverter_abutted(gf180, Wp=6.0, Wn=3.0, Lp=0.36, Ln=0.36)
inverter_layout.write_gds(ruta_gds_inverter)
print(f"Guardado en: {ruta_gds_inverter}")
#klayout_viewer(ruta_gds_inverter)
#inverter_layout.show()
run_drc_v2(ruta_gds_inverter)

/tmp/ipykernel_42142/4244995193.py:146: UserWarning: Unnamed cells, 6 in 'inv_layout_1786940610'
  inverter_layout.write_gds(ruta_gds_inverter)
2026-08-17 06:23:34.889 | INFO     | gdsfactory.component:_write_library:1851 - Wrote to '/foss/designs/layout/inverter.gds'


Guardado en: /foss/designs/layout/inverter.gds
?? Ejecutando DRC para: /foss/designs/layout/inverter.gds ...
¿Diseño libre de errores (DRC Clean)?: False

----- DRC STDOUT -----
2026-08-17 06:23:38 +0200: Memory Usage (448220K) : Starting running GF180MCU Klayout DRC runset on /foss/designs/layout/inverter.gds
2026-08-17 06:23:38 +0200: Memory Usage (448220K) : Ruby Version for klayout: 3.2.3
2026-08-17 06:23:38 +0200: Memory Usage (449132K) : Loading database to memory is complete.
2026-08-17 06:23:38 +0200: Memory Usage (449132K) : GF180MCU Klayout DRC runset output at: /foss/designs/layout/inverter_main.lyrdb
2026-08-17 06:23:38 +0200: Memory Usage (449276K) : Evaluate switches.
2026-08-17 06:23:38 +0200: Memory Usage (449276K) : table_name selected  main
2026-08-17 06:23:38 +0200: Memory Usage (449276K) : CONNECTIVITY_RULES enabled: true
2026-08-17 06:23:38 +0200: Memory Usage (449276K) : Wedge enabled: false
2026-08-17 06:23:38 +0200: Memory Usage (449276K) : Ball enabled: false
2

ERROR: In /foss/pdks/ciel/gf180mcu/versions/7b70722e33c03fcb5dabcf4d479fb0822d9251c9/gf180mcuD/libs.tech/klayout/tech/macros/gf180mcu_lvs.lylvs: Can't find a schematic counterpart for the top cell inv_layout_1786940610 - use 'same_circuit' to establish a correspondence
ERROR: :/built-in-macros/_lvs_netter.rb:302: RuntimeError: Can't find a schematic counterpart for the top cell inv_layout_1786940610 - use 'same_circuit' to establish a correspondence in Executable::execute
  :/built-in-macros/_lvs_netter.rb:302:in `align'
  (eval):2:in `align'
  /foss/pdks/ciel/gf180mcu/versions/7b70722e33c03fcb5dabcf4d479fb0822d9251c9/gf180mcuD/libs.tech/klayout/tech/macros/../lvs/gf180mcu.lvs:358:in `execute'
  :/built-in-macros/lvs_interpreters.lym:31:in `instance_eval'
  :/built-in-macros/lvs_interpreters.lym:31:in `execute'


2026-08-17 06:23:57 +0200: Memory Usage (3274600K) : Starting GF180 LVS comparison section
2026-08-17 06:24:07 +0200: Memory Usage (3281776K) : Starting running GF180MCU Klayout LVS runset on 
2026-08-17 06:24:07 +0200: Memory Usage (3281776K) : Ruby Version for klayout: 3.2.3
2026-08-17 06:24:07 +0200: Memory Usage (3281776K) : Loading database to memory is complete.
2026-08-17 06:24:07 +0200: Memory Usage (3281776K) : GF180MCU Klayout LVS runset output at default location: inverter.lvsdb
2026-08-17 06:24:07 +0200: Memory Usage (3281776K) : Evaluate switches.
2026-08-17 06:24:07 +0200: Memory Usage (3281776K) : Substrate name used: gf180mcu_gnd
2026-08-17 06:24:07 +0200: Memory Usage (3281776K) : Extracted netlist with net names: true
2026-08-17 06:24:07 +0200: Memory Usage (3281776K) : Extracted netlist with comments in details: false
2026-08-17 06:24:07 +0200: Memory Usage (3281776K) : GF180MCU Klayout LVS extracted netlist file at: inverter_extracted.cir
2026-08-17 06:24:07 +0200: 

In [62]:
verificar_drc_manual("inverter.gds")

?? Ejecutando DRC para: /foss/designs/layout/inverter.gds ...
?? DRC finalizó con errores de ejecución (Revisar logs).
?? Reporte de DRC detectado en: /foss/designs/layout/inverter_main.lyrdb
?? Limpiando memoria de KLayout...
?? Abriendo KLayout...
2026-08-21 06:32:55 +0200: Memory Usage (3107632K) : Starting running GF180MCU Klayout LVS runset on 
2026-08-21 06:32:55 +0200: Memory Usage (3107632K) : Ruby Version for klayout: 3.2.3
2026-08-21 06:32:55 +0200: Memory Usage (3107632K) : Loading database to memory is complete.
2026-08-21 06:32:55 +0200: Memory Usage (3107632K) : GF180MCU Klayout LVS runset output at default location: inverter.lvsdb
2026-08-21 06:32:55 +0200: Memory Usage (3107632K) : Evaluate switches.
2026-08-21 06:32:55 +0200: Memory Usage (3107632K) : Substrate name used: gf180mcu_gnd
2026-08-21 06:32:55 +0200: Memory Usage (3107632K) : Extracted netlist with net names: true
2026-08-21 06:32:55 +0200: Memory Usage (3107632K) : Extracted netlist with comments in details

In [33]:
#Layout espejado en el eje X comparado con el esquematico
import time
from gdsfactory import Component
from glayout.flow.primitives.fet import nmos, pmos
from glayout.flow.primitives.guardring import tapring
from glayout.flow.routing.smart_route import smart_route
from glayout.flow.routing.c_route import c_route
from glayout.flow.routing.straight_route import straight_route
from glayout.flow.pdk.gf180_mapped import gf180_mapped_pdk as gf180

def create_cmfb_circuit(pdk_config):
    top_level = Component(f"cmfb_circuit_{int(time.time())}")

    # ==========================================
    # 1. Configuración de Celdas
    # ==========================================
    # Forzamos TODO a met1 nativamente para control absoluto.
    pfet_kwargs = {
        "fingers": 1, 
        "with_tie": False, 
        "with_substrate_tap": False, 
        "with_dummy": False,
        "dnwell": False,         
        "sd_route_topmet": "met3",    
        "gate_route_topmet": "met2"   
    }
    
    nfet_kwargs = {
        "fingers": 1, 
        "with_tie": False, 
        "with_substrate_tap": False, 
        "with_dummy": False,
        "with_dnwell": False,         
        "sd_route_topmet": "met3",    
        "gate_route_topmet": "met2"   
    }

    # Fila Superior: Cargas PMOS
    xm20 = top_level << pmos(pdk_config, width=3.0, length=1.0, **pfet_kwargs)
    xm18 = top_level << pmos(pdk_config, width=3.0, length=1.0, **pfet_kwargs)
    xm17 = top_level << pmos(pdk_config, width=3.0, length=1.0, **pfet_kwargs)

    # Fila Media: Pares Diferenciales NMOS
    xm21 = top_level << nmos(pdk_config, width=10.0, length=1.0, **nfet_kwargs)
    xm22 = top_level << nmos(pdk_config, width=10.0, length=1.0, **nfet_kwargs)
    xm24 = top_level << nmos(pdk_config, width=10.0, length=1.0, **nfet_kwargs)
    xm25 = top_level << nmos(pdk_config, width=10.0, length=1.0, **nfet_kwargs)

    # Fila Inferior: Espejos de Corriente NMOS
    xm27 = top_level << nmos(pdk_config, width=8.0, length=1.0, **nfet_kwargs)
    xm23 = top_level << nmos(pdk_config, width=8.0, length=1.0, **nfet_kwargs)
    xm26 = top_level << nmos(pdk_config, width=8.0, length=1.0, **nfet_kwargs)

    # ==========================================
    # 2. Floorplan Expandido (DRC Clearance)
    # ==========================================
    x_center = 0.0
    x_inner = 4.5   # Aumentado para dar espacio a ruteo vertical central
    x_outer = 9.5   # Aumentado para dar espacio a ruteo exterior

    y_gap_nmos = 4.0  # Espacio para pistas horizontales intermedias
    y_gap_n2p = 10.0  # Gran espacio para separar NWELL y PWELL rings

    # Ubicación Fila Inferior
    xm27.movex(x_center)
    xm23.movex(-x_outer)
    xm26.movex(x_outer)
    xm23.movey(xm27.ymin - xm23.ymin)
    xm26.movey(xm27.ymin - xm26.ymin)

    # Ubicación Fila Media
    row2_y = xm27.ymax + y_gap_nmos
    xm22.movex(-x_inner)
    xm24.movex(x_inner)
    xm21.movex(-x_outer)  
    xm25.movex(x_outer)   

    for inst in [xm21, xm22, xm24, xm25]:
        inst.movey(row2_y - inst.ymin)

    # Ubicación Fila Superior
    row3_y = xm22.ymax + y_gap_n2p
    xm18.movex(x_center)
    xm20.movex(-x_outer)  
    xm17.movex(x_outer)   

    for inst in [xm18, xm20, xm17]:
        inst.movey(row3_y - inst.ymin)

    # ==========================================
    # 3. Tap Rings y Pozos (Adaptados al nuevo tamaño)
    # ==========================================
    tap_margin = 1.8 

    # NMOS (P-Tap)
    nmos_xmin = min(xm21.xmin, xm23.xmin, xm25.xmin, xm26.xmin) - tap_margin
    nmos_xmax = max(xm21.xmax, xm23.xmax, xm25.xmax, xm26.xmax) + tap_margin
    nmos_ymin = min(xm27.ymin, xm23.ymin, xm26.ymin) - tap_margin
    nmos_ymax = max(xm21.ymax, xm22.ymax, xm24.ymax, xm25.ymax) + tap_margin
    
    ptap = top_level << tapring(
        pdk_config, enclosed_rectangle=(nmos_xmax - nmos_xmin, nmos_ymax - nmos_ymin), sdlayer="p+s/d"
    )
    ptap.move((nmos_xmin - ptap.xmin, nmos_ymin - ptap.ymin))

    if "pwell" in pdk_config.glayers:
        top_level.add_polygon(
            [[ptap.xmin, ptap.ymin], [ptap.xmax, ptap.ymin], [ptap.xmax, ptap.ymax], [ptap.xmin, ptap.ymax]],
            layer=pdk_config.glayers["pwell"]
        )

    # PMOS (N-Tap)
    pmos_xmin = nmos_xmin 
    pmos_xmax = nmos_xmax
    pmos_ymin = xm18.ymin - tap_margin
    pmos_ymax = xm18.ymax + tap_margin
    
    ntap = top_level << tapring(
        pdk_config, enclosed_rectangle=(pmos_xmax - pmos_xmin, pmos_ymax - pmos_ymin), sdlayer="n+s/d"
    )
    ntap.move((pmos_xmin - ntap.xmin, pmos_ymin - ntap.ymin))

    nwell_margin = 0.6
    top_level.add_polygon(
        [
            [ntap.xmin - nwell_margin, ntap.ymin - nwell_margin],
            [ntap.xmax + nwell_margin, ntap.ymin - nwell_margin],
            [ntap.xmax + nwell_margin, ntap.ymax + nwell_margin],
            [ntap.xmin - nwell_margin, ntap.ymax + nwell_margin]
        ],
        layer=pdk_config.glayers["nwell"]
    )

    # ==========================================
    # 4. Enrutamiento Inteligente (Fijando M1, M2, MT)
    # ==========================================
    # Conexiones Diode-connected
    top_level << smart_route(pdk_config, xm20.ports['drain_E'], xm20.ports['gate_E'])
    top_level << smart_route(pdk_config, xm18.ports['drain_E'], xm18.ports['gate_E'], extension=1.5)
    top_level << smart_route(pdk_config, xm17.ports['drain_E'], xm17.ports['gate_E'])
    top_level << smart_route(pdk_config, xm27.ports['drain_E'], xm27.ports['gate_E'])

    # Redes horizontales (Strictamente met1 a través de straight_route)
    top_level << straight_route(pdk_config, xm23.ports['source_E'], xm27.ports['source_W'])
    top_level << straight_route(pdk_config, xm27.ports['source_E'], xm26.ports['source_W'])
    top_level << straight_route(pdk_config, xm20.ports['source_E'], xm18.ports['source_W'])
    top_level << straight_route(pdk_config, xm18.ports['source_E'], xm17.ports['source_W'])

    top_level << straight_route(pdk_config, xm23.ports['gate_E'], xm27.ports['gate_W'])
    top_level << straight_route(pdk_config, xm27.ports['gate_E'], xm26.ports['gate_W'])
    top_level << straight_route(pdk_config, xm22.ports['gate_E'], xm24.ports['gate_W'])

    top_level << straight_route(pdk_config, xm21.ports['source_E'], xm22.ports['source_W'])
    top_level << straight_route(pdk_config, xm24.ports['source_E'], xm25.ports['source_W'])

    # ==========================================
    # RUTEO ORTOGONAL ESCALONADO (Usando met2 para evitar MT errors)
    # ==========================================
    # Espejos a Pares Diferenciales
    top_level << c_route(
        pdk_config, xm23.ports['drain_W'], xm21.ports['source_W'], 
        extension=4.5, e1glayer="met3", e2glayer="met3", cglayer="met2"
    )
    
    top_level << c_route(
        pdk_config, xm26.ports['drain_E'], xm25.ports['source_E'], 
        extension=4.5, e1glayer="met3", e2glayer="met3", cglayer="met2"
    )

    # Drains NMOS a Drains PMOS (Rutas Exteriores) - Extension gigante
    top_level << c_route(
        pdk_config, xm21.ports['drain_W'], xm20.ports['drain_W'], 
        extension=4.5, e1glayer="met3", e2glayer="met3", cglayer="met4"
    )
    
    top_level << c_route(
        pdk_config, xm25.ports['drain_E'], xm17.ports['drain_E'], 
        extension=4.5, e1glayer="met3", e2glayer="met3", cglayer="met4"
    )
    
    # Drains NMOS a Drains PMOS (Rutas Interiores) - Escalonadas
    top_level << c_route(
        pdk_config, xm22.ports['drain_E'], xm18.ports['drain_E'], 
        extension=5.5, e1glayer="met3", e2glayer="met3", cglayer="met4"
    )
    
    top_level << c_route(
        pdk_config, xm24.ports['drain_W'], xm18.ports['drain_W'], 
        extension=4.5, e1glayer="met3", e2glayer="met3", cglayer="met4"
    )

    # Limpieza final LVPWELL
    top_level = top_level.remove_layers(layers=[(204, 0)])

    return top_level

if __name__ == "__main__":
    cmfb_layout = create_cmfb_circuit(gf180)
    ruta_gds_cmfb = "cmfb.gds"
    cmfb_layout.write_gds(ruta_gds_cmfb)
    print(f"GDS Generado Exitosamente: {ruta_gds_cmfb}")
    run_drc_v2(ruta_gds_cmfb)

(22, 0)
## gf180mcu PDK Pcells loaded.
['/foss/pdks/ciel/gf180mcu/versions/7b70722e33c03fcb5dabcf4d479fb0822d9251c9/gf180mcuD/libs.tech/klayout/tech/pymacros', '/foss/tools/klayout/pymod', '/foss/tools/klayout_gdsfactory9/lib/python3.12/site-packages', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '/headless/.local/lib/python3.12/site-packages', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/headless/.klayout/python', '/headless/.klayout/salt/KLayoutPluginUtils/python', '/headless/.klayout/salt/klive/python']
klive 0.4.1 is running


/tmp/ipykernel_913/4056876811.py:200: UserWarning: Unnamed cells, 1 in 'Unnamed_ce0cf39b'
  cmfb_layout.write_gds(ruta_gds_cmfb)
2026-08-18 06:16:52.146 | INFO     | gdsfactory.component:_write_library:1851 - Wrote to 'cmfb.gds'


GDS Generado Exitosamente: cmfb.gds
?? Ejecutando DRC para: cmfb.gds ...
¿Diseño libre de errores (DRC Clean)?: False

----- DRC STDOUT -----
2026-08-18 06:16:54 +0200: Memory Usage (448280K) : Starting running GF180MCU Klayout DRC runset on /foss/designs/layout/cmfb.gds
2026-08-18 06:16:54 +0200: Memory Usage (448280K) : Ruby Version for klayout: 3.2.3
2026-08-18 06:16:54 +0200: Memory Usage (449284K) : Loading database to memory is complete.
2026-08-18 06:16:54 +0200: Memory Usage (449284K) : GF180MCU Klayout DRC runset output at: /foss/designs/layout/cmfb_main.lyrdb
2026-08-18 06:16:54 +0200: Memory Usage (449412K) : Evaluate switches.
2026-08-18 06:16:54 +0200: Memory Usage (449412K) : table_name selected  main
2026-08-18 06:16:54 +0200: Memory Usage (449412K) : CONNECTIVITY_RULES enabled: true
2026-08-18 06:16:54 +0200: Memory Usage (449412K) : Wedge enabled: false
2026-08-18 06:16:54 +0200: Memory Usage (449412K) : Ball enabled: false
2026-08-18 06:16:54 +0200: Memory Usage (4494

ERROR: In /foss/pdks/ciel/gf180mcu/versions/7b70722e33c03fcb5dabcf4d479fb0822d9251c9/gf180mcuD/libs.tech/klayout/tech/macros/gf180mcu_lvs.lylvs: Can't find a schematic counterpart for the top cell Unnamed_ce0cf39b - use 'same_circuit' to establish a correspondence
ERROR: :/built-in-macros/_lvs_netter.rb:302: RuntimeError: Can't find a schematic counterpart for the top cell Unnamed_ce0cf39b - use 'same_circuit' to establish a correspondence in Executable::execute
  :/built-in-macros/_lvs_netter.rb:302:in `align'
  (eval):2:in `align'
  /foss/pdks/ciel/gf180mcu/versions/7b70722e33c03fcb5dabcf4d479fb0822d9251c9/gf180mcuD/libs.tech/klayout/tech/macros/../lvs/gf180mcu.lvs:358:in `execute'
  :/built-in-macros/lvs_interpreters.lym:31:in `instance_eval'
  :/built-in-macros/lvs_interpreters.lym:31:in `execute'


2026-08-18 06:24:02 +0200: Memory Usage (3595344K) : Starting GF180 LVS comparison section
2026-08-18 06:24:14 +0200: Memory Usage (3597700K) : Starting running GF180MCU Klayout LVS runset on 
2026-08-18 06:24:14 +0200: Memory Usage (3597700K) : Ruby Version for klayout: 3.2.3
2026-08-18 06:24:14 +0200: Memory Usage (3597700K) : Loading database to memory is complete.
2026-08-18 06:24:14 +0200: Memory Usage (3597700K) : GF180MCU Klayout LVS runset output at default location: cmfb.lvsdb
2026-08-18 06:24:14 +0200: Memory Usage (3597700K) : Evaluate switches.
2026-08-18 06:24:14 +0200: Memory Usage (3597700K) : Substrate name used: gf180mcu_gnd
2026-08-18 06:24:14 +0200: Memory Usage (3597700K) : Extracted netlist with net names: true
2026-08-18 06:24:14 +0200: Memory Usage (3597700K) : Extracted netlist with comments in details: false
2026-08-18 06:24:14 +0200: Memory Usage (3597700K) : GF180MCU Klayout LVS extracted netlist file at: cmfb_extracted.cir
2026-08-18 06:24:14 +0200: Memory U

In [64]:
verificar_drc_manual("cmfb.gds")

?? Ejecutando DRC para: /foss/designs/layout/cmfb.gds ...
?? DRC finalizó con errores de ejecución (Revisar logs).
?? Reporte de DRC detectado en: /foss/designs/layout/cmfb_main.lyrdb
?? Limpiando memoria de KLayout...
?? Abriendo KLayout...
2026-08-21 06:37:11 +0200: Memory Usage (3108016K) : Starting running GF180MCU Klayout LVS runset on 
2026-08-21 06:37:11 +0200: Memory Usage (3108016K) : Ruby Version for klayout: 3.2.3
2026-08-21 06:37:11 +0200: Memory Usage (3108016K) : Loading database to memory is complete.
2026-08-21 06:37:11 +0200: Memory Usage (3108016K) : GF180MCU Klayout LVS runset output at default location: cmfb.lvsdb
2026-08-21 06:37:11 +0200: Memory Usage (3108016K) : Evaluate switches.
2026-08-21 06:37:11 +0200: Memory Usage (3108016K) : Substrate name used: gf180mcu_gnd
2026-08-21 06:37:11 +0200: Memory Usage (3108016K) : Extracted netlist with net names: true
2026-08-21 06:37:11 +0200: Memory Usage (3108016K) : Extracted netlist with comments in details: false
2026

In [71]:
import time
from gdsfactory import Component
from glayout.flow.pdk.gf180_mapped import gf180_mapped_pdk as gf180
from glayout.flow.primitives.fet import nmos, pmos
from glayout.flow.primitives.guardring import tapring
from glayout.flow.routing.smart_route import smart_route

# Bias Circuit v2
def create_bias_circuit(pdk_config):
    
    # Usamos time.time() para evitar superposición en caché de KLayout
    nombre_seguro = f"bias_circuit_{int(time.time())}"
    top_level = Component(nombre_seguro)

    # ==========================================
    # 1. Generación de Celdas (Separación en Z)
    # ==========================================
    fet_kwargs_nmos = {
        "fingers": 1, 
        "with_tie": False, 
        "with_substrate_tap": False, 
        "with_dummy": False,
        "with_dnwell": False,
        "sd_route_topmet": "met3",
        "gate_route_topmet": "met2"
    }

    fet_kwargs_pmos = {
        "fingers": 1, 
        "with_tie": False, 
        "with_substrate_tap": False, 
        "with_dummy": False,
        "dnwell": False,
        "sd_route_topmet": "met3",
        "gate_route_topmet": "met2"
    }

    xm12 = top_level << nmos(pdk_config, width=1.0, length=1.0, **fet_kwargs_nmos)
    xm13 = top_level << nmos(pdk_config, width=3.0, length=1.0, **fet_kwargs_nmos)
    xm15 = top_level << nmos(pdk_config, width=4.0, length=1.0, **fet_kwargs_nmos)

    xm14 = top_level << pmos(pdk_config, width=6.0, length=1.0, **fet_kwargs_pmos)
    xm16 = top_level << pmos(pdk_config, width=11.0, length=1.0, **fet_kwargs_pmos)
    xm28 = top_level << pmos(pdk_config, width=4.9, length=1.0, **fet_kwargs_pmos)

    # ==========================================
    # 2. Posicionamiento (Ultra Compacto y Alineado)
    # ==========================================
    y_spacing = 3.0  
    x_spacing = 6.0 

    # Apilamiento NMOS
    xm13.ymin = xm12.ymax + y_spacing
    xm13.x = xm12.x
    xm15.ymin = xm13.ymax + y_spacing
    xm15.x = xm12.x

    # Apilamiento PMOS (Alineado centralmente con NMOS)
    xm14.y = xm13.y
    xm14.xmin = xm12.xmax + x_spacing

    xm16.ymin = xm14.ymax + y_spacing
    xm16.x = xm14.x

    xm28.ymax = xm14.ymin - y_spacing
    xm28.x = xm14.x

    # ==========================================
    # 3. Anillos de Sustrato y NWELL Global
    # ==========================================
    # --- Bloque NMOS ---
    nmos_xmin = min(xm12.xmin, xm13.xmin, xm15.xmin)
    nmos_xmax = max(xm12.xmax, xm13.xmax, xm15.xmax)
    nmos_ymin = min(xm12.ymin, xm13.ymin, xm15.ymin)
    nmos_ymax = max(xm12.ymax, xm13.ymax, xm15.ymax)
    
    nmos_margin = 2.5
    ptap_w = (nmos_xmax - nmos_xmin) + nmos_margin
    ptap_h = (nmos_ymax - nmos_ymin) + nmos_margin
    
    ptap = top_level << tapring(pdk_config, enclosed_rectangle=(ptap_w, ptap_h), sdlayer="p+s/d")
    ptap.x = (nmos_xmin + nmos_xmax) / 2
    ptap.y = (nmos_ymin + nmos_ymax) / 2

    # --- Bloque PMOS ---
    pmos_xmin = min(xm14.xmin, xm16.xmin, xm28.xmin)
    pmos_xmax = max(xm14.xmax, xm16.xmax, xm28.xmax)
    pmos_ymin = min(xm14.ymin, xm16.ymin, xm28.ymin)
    pmos_ymax = max(xm14.ymax, xm16.ymax, xm28.ymax)
    
    pmos_margin = 2.5
    pmos_w = (pmos_xmax - pmos_xmin) + pmos_margin
    pmos_h = (pmos_ymax - pmos_ymin) + pmos_margin

    ntap = top_level << tapring(pdk_config, enclosed_rectangle=(pmos_w, pmos_h), sdlayer="n+s/d")
    ntap.x = (pmos_xmin + pmos_xmax) / 2
    ntap.y = (pmos_ymin + pmos_ymax) / 2

    # NWELL Global
    nwell_overlap = 0.5
    top_level.add_polygon(
        [
            [ntap.xmin - nwell_overlap, ntap.ymin - nwell_overlap],
            [ntap.xmax + nwell_overlap, ntap.ymin - nwell_overlap],
            [ntap.xmax + nwell_overlap, ntap.ymax + nwell_overlap],
            [ntap.xmin - nwell_overlap, ntap.ymax + nwell_overlap]
        ],
        layer=pdk_config.glayers["nwell"]
    )

    # ==========================================
    # 4. Enrutamiento del Esquemático Completo
    # ==========================================
    
    # Diodos (Lazos Locales)
    top_level << smart_route(pdk_config, xm12.ports['drain_E'], xm12.ports['gate_E'])
    top_level << smart_route(pdk_config, xm14.ports['drain_E'], xm14.ports['gate_E'])
    top_level << smart_route(pdk_config, xm15.ports['drain_W'], xm15.ports['gate_W'])

    # Espejos / Buses de Gate Verticales
    top_level << smart_route(pdk_config, xm12.ports['gate_W'], xm13.ports['gate_W'])
    top_level << smart_route(pdk_config, xm14.ports['gate_W'], xm16.ports['gate_W'])
    top_level << smart_route(pdk_config, xm28.ports['gate_W'], xm14.ports['gate_W'])

    # --- CORRECCIÓN AQUÍ: Cruces Horizontales Drain-Drain ---
    # Igualamos las orientaciones (ambas a 'drain_E') para cumplir con las reglas de c_route.
    # Asignamos cglayer="met3" explícitamente para evitar los errores DRC de TopMetal.
    top_level << smart_route(pdk_config, xm13.ports['drain_E'], xm14.ports['drain_E'], cglayer="met3", e1glayer="met4")  
    top_level << smart_route(pdk_config, xm15.ports['drain_E'], xm16.ports['drain_E'], cglayer="met3")

    # Alimentación (Sources Verticales)
    top_level << smart_route(pdk_config, xm12.ports['source_W'], xm13.ports['source_W'])
    top_level << smart_route(pdk_config, xm13.ports['source_W'], xm15.ports['source_W'])
    
    top_level << smart_route(pdk_config, xm28.ports['source_E'], xm14.ports['source_E'])
    top_level << smart_route(pdk_config, xm14.ports['source_E'], xm16.ports['source_E'])

    # Limpieza de capas de enrutamiento
    top_level = top_level.remove_layers(layers=[(204, 0)])

    # ==========================================
    # 5. Pines Externos
    # ==========================================
    top_level.add_ports(xm12.get_ports_list(prefix="1ua_"), prefix="1ua_")
    top_level.add_ports(xm28.get_ports_list(prefix="ib_"), prefix="ib_")
    top_level.add_ports(xm14.get_ports_list(prefix="Vbtail_"), prefix="Vbtail_")
    top_level.add_ports(xm16.get_ports_list(prefix="Vbsnk_"), prefix="Vbsnk_")
    
    top_level.add_ports(ptap.get_ports_list(prefix="vss_"), prefix="vss_")
    top_level.add_ports(ntap.get_ports_list(prefix="vdd_"), prefix="vdd_")

    return top_level


if __name__ == "__main__":
    bias_layout = create_bias_circuit(gf180)
    ruta_gds_bias = "/foss/designs/layout/bias.gds"
    bias_layout.write_gds(ruta_gds_bias)
    
    # Asumo que run_drc_v2 es una función definida en tu entorno, la dejo intacta:
    try:
        run_drc_v2(ruta_gds_bias)
    except NameError:
        print("La función run_drc_v2 no está definida en este contexto de ejecución.")


/tmp/ipykernel_931/2751765777.py:158: UserWarning: Unnamed cells, 1 in 'Unnamed_46e1347c'
  bias_layout.write_gds(ruta_gds_bias)
2026-08-21 06:50:01.498 | INFO     | gdsfactory.component:_write_library:1851 - Wrote to '/foss/designs/layout/bias.gds'


?? Ejecutando DRC para: /foss/designs/layout/bias.gds ...
¿Diseño libre de errores (DRC Clean)?: False

----- DRC STDOUT -----
2026-08-21 06:50:04 +0200: Memory Usage (448952K) : Starting running GF180MCU Klayout DRC runset on /foss/designs/layout/bias.gds
2026-08-21 06:50:04 +0200: Memory Usage (448952K) : Ruby Version for klayout: 3.2.3
2026-08-21 06:50:04 +0200: Memory Usage (449960K) : Loading database to memory is complete.
2026-08-21 06:50:04 +0200: Memory Usage (449960K) : GF180MCU Klayout DRC runset output at: /foss/designs/layout/bias_main.lyrdb
2026-08-21 06:50:04 +0200: Memory Usage (450088K) : Evaluate switches.
2026-08-21 06:50:04 +0200: Memory Usage (450088K) : table_name selected  main
2026-08-21 06:50:04 +0200: Memory Usage (450088K) : CONNECTIVITY_RULES enabled: true
2026-08-21 06:50:04 +0200: Memory Usage (450088K) : Wedge enabled: false
2026-08-21 06:50:04 +0200: Memory Usage (450088K) : Ball enabled: false
2026-08-21 06:50:04 +0200: Memory Usage (450088K) : Gold ena

In [74]:
verificar_drc_manual("bias.gds")

?? Ejecutando DRC para: /foss/designs/layout/bias.gds ...
?? DRC finalizó con errores de ejecución (Revisar logs).
?? Reporte de DRC detectado en: /foss/designs/layout/bias_main.lyrdb
?? Limpiando memoria de KLayout...
?? Abriendo KLayout...
(22, 0)
## gf180mcu PDK Pcells loaded.
['/foss/pdks/ciel/gf180mcu/versions/7b70722e33c03fcb5dabcf4d479fb0822d9251c9/gf180mcuD/libs.tech/klayout/tech/pymacros', '/foss/tools/klayout/pymod', '/foss/tools/klayout_gdsfactory9/lib/python3.12/site-packages', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '/headless/.local/lib/python3.12/site-packages', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/headless/.klayout/python', '/headless/.klayout/salt/KLayoutPluginUtils/python', '/headless/.klayout/salt/klive/python']
klive 0.4.1 is running


In [52]:
# LAYOUT DE BUFFER - DRC CLEAN Y SIN DESPERDICIO DE ÁREA

from glayout.flow.pdk.mappedpdk import MappedPDK
from glayout.flow.primitives.fet import nmos, pmos
from glayout.flow.primitives.guardring import tapring
from glayout.flow.routing.straight_route import straight_route
from glayout.flow.routing.c_route import c_route
from gdsfactory import Component

def buffer_pro(
    pdk: MappedPDK,
    w_pmos=1.0, l_pmos=1.0, f_pmos=6,
    w_nmos_diff=1.0, l_nmos_diff=1.0, f_nmos_diff=6,
    w_nmos_tail=1.0, l_nmos_tail=1.0, f_nmos_tail=10
):
    comp = Component("buffer_drcclean_optimo")

    # 1. PARÁMETROS LIMPIOS
    nmos_kwargs = {
        "with_tie": False, "with_substrate_tap": False, "with_dummy": False, 
        "with_dnwell": False, "dummy_routes": False,
        "sd_route_topmet": "met3", "gate_route_topmet": "met2"
    }
    
    pmos_kwargs = {
        "with_tie": False, "with_substrate_tap": False, "with_dummy": False, 
        "dnwell": False, "dummy_routes": False,
        "sd_route_topmet": "met3", "gate_route_topmet": "met2"
    }

    # 2. GENERACIÓN
    p_l = comp << pmos(pdk, width=w_pmos, length=l_pmos, fingers=f_pmos, **pmos_kwargs)
    p_r = comp << pmos(pdk, width=w_pmos, length=l_pmos, fingers=f_pmos, **pmos_kwargs)
    n_l = comp << nmos(pdk, width=w_nmos_diff, length=l_nmos_diff, fingers=f_nmos_diff, **nmos_kwargs)
    n_r = comp << nmos(pdk, width=w_nmos_diff, length=l_nmos_diff, fingers=f_nmos_diff, **nmos_kwargs)
    n_t = comp << nmos(pdk, width=w_nmos_tail, length=l_nmos_tail, fingers=f_nmos_tail, **nmos_kwargs)

    # 3. UBICACIÓN
    offset_x = 6.0  
    y_pmos = 12.0   
    y_diff = -1.0   
    y_tail = -9.0   

    p_l.move(origin=p_l.center, destination=(-offset_x, y_pmos))
    p_r.move(origin=p_r.center, destination=(offset_x, y_pmos))
    n_l.move(origin=n_l.center, destination=(-offset_x, y_diff))
    n_r.move(origin=n_r.center, destination=(offset_x, y_diff))
    n_t.move(origin=n_t.center, destination=(0, y_tail))

    # 4. RUTEO DE MACRO
    w_route = 0.35
    ext_internal = 2.0
    
    comp << straight_route(pdk, p_l.ports["gate_E"], p_r.ports["gate_W"], glayer1="met2", width=w_route)
    comp << c_route(pdk, p_l.ports["gate_W"], p_l.ports["drain_W"], extension=ext_internal, e1glayer="met2", e2glayer="met3", cglayer="met3", width1=w_route, width2=w_route, cwidth=w_route)
    comp << c_route(pdk, n_r.ports["drain_E"], n_r.ports["gate_E"], extension=ext_internal, e1glayer="met3", e2glayer="met2", cglayer="met3", width1=w_route, width2=w_route, cwidth=w_route)
    comp << straight_route(pdk, p_l.ports["drain_S"], n_l.ports["drain_N"], glayer1="met3", width=w_route)
    comp << straight_route(pdk, p_r.ports["drain_S"], n_r.ports["drain_N"], glayer1="met3", width=w_route)
    comp << c_route(pdk, n_l.ports["source_W"], n_t.ports["drain_W"], extension=ext_internal, e1glayer="met3", e2glayer="met3", cglayer="met3", width1=w_route, width2=w_route, cwidth=w_route)
    comp << c_route(pdk, n_r.ports["source_E"], n_t.ports["drain_E"], extension=ext_internal, e1glayer="met3", e2glayer="met3", cglayer="met3", width1=w_route, width2=w_route, cwidth=w_route)

    # 5. ANILLOS DE GUARDA (Cajas ajustadas al mínimo)
    margin = 1.5 
    y_shift = 0.8  # Desfase posicional sutil para salvar el DRC en PMOS

    # Tap Ring NWELL (Tamaño ajustado, solo movemos el centro hacia abajo)
    p_xmin, p_xmax = p_l.xmin, p_r.xmax
    p_ymin, p_ymax = p_l.ymin, p_r.ymax
    enclosed_p_w = (p_xmax - p_xmin) + 2*margin
    enclosed_p_h = (p_ymax - p_ymin) + 2*margin  # ¡Sin áreas extra!
    ptap = comp << tapring(pdk, enclosed_rectangle=(enclosed_p_w, enclosed_p_h), sdlayer="n+s/d")
    ptap.move(origin=ptap.center, destination=((p_xmin + p_xmax) / 2.0, ((p_ymin + p_ymax) / 2.0) - y_shift))

    # NWELL Box manual
    nw_margin = 0.5
    comp.add_polygon([
        (ptap.xmin - nw_margin, ptap.ymin - nw_margin), (ptap.xmax + nw_margin, ptap.ymin - nw_margin),
        (ptap.xmax + nw_margin, ptap.ymax + nw_margin), (ptap.xmin - nw_margin, ptap.ymax + nw_margin)
    ], layer=pdk.get_glayer("nwell"))

    # Tap Ring P-SUB (Tamaño ajustado y centrado natural)
    n_xmin, n_xmax = min(n_l.xmin, n_t.xmin), max(n_r.xmax, n_t.xmax)
    n_ymin, n_ymax = n_t.ymin, n_l.ymax
    enclosed_n_w = (n_xmax - n_xmin) + 2*margin
    enclosed_n_h = (n_ymax - n_ymin) + 2*margin  # ¡Sin áreas extra!
    ntap = comp << tapring(pdk, enclosed_rectangle=(enclosed_n_w, enclosed_n_h), sdlayer="p+s/d")
    ntap.move(origin=ntap.center, destination=((n_xmin + n_xmax) / 2.0, (n_ymin + n_ymax) / 2.0))

    # 6. CONEXIÓN A LOS RINGS
    ext_ring = 2.0 
    comp << c_route(pdk, p_l.ports["source_W"], ptap.ports["W_top_met_W"], extension=ext_ring, e1glayer="met3", e2glayer="met1", cglayer="met3", width1=w_route, width2=w_route, cwidth=w_route)
    comp << c_route(pdk, p_r.ports["source_E"], ptap.ports["E_top_met_E"], extension=ext_ring, e1glayer="met3", e2glayer="met1", cglayer="met3", width1=w_route, width2=w_route, cwidth=w_route)
    comp << c_route(pdk, n_t.ports["source_W"], ntap.ports["W_top_met_W"], extension=ext_ring, e1glayer="met3", e2glayer="met1", cglayer="met3", width1=w_route, width2=w_route, cwidth=w_route)
    comp << c_route(pdk, n_t.ports["source_E"], ntap.ports["E_top_met_E"], extension=ext_ring, e1glayer="met3", e2glayer="met1", cglayer="met3", width1=w_route, width2=w_route, cwidth=w_route)

    return comp.remove_layers(layers=[(204, 0)])

# --- Ejecución ---
from glayout.flow.pdk.gf180_mapped import gf180_mapped_pdk as gf180 
ruta_gds_buffer = "/foss/designs/layout/buffer.gds"
layout_final = buffer_pro(gf180)
layout_final.write_gds(ruta_gds)
print(f"Layout optimizado en área generado en: {ruta_gds}")

run_drc_v2(ruta_gds_buffer)

/tmp/ipykernel_931/4091867196.py:102: UserWarning: Unnamed cells, 1 in 'Unnamed_c759c5b9'
  layout_final.write_gds(ruta_gds)
2026-08-21 05:42:27.463 | INFO     | gdsfactory.component:_write_library:1851 - Wrote to '/foss/designs/layout/buffer.gds'


Layout optimizado en área generado en: /foss/designs/layout/buffer.gds
?? Ejecutando DRC para: /foss/designs/layout/buffer.gds ...
¿Diseño libre de errores (DRC Clean)?: False

----- DRC STDOUT -----
2026-08-21 05:42:31 +0200: Memory Usage (449084K) : Starting running GF180MCU Klayout DRC runset on /foss/designs/layout/buffer.gds
2026-08-21 05:42:31 +0200: Memory Usage (449084K) : Ruby Version for klayout: 3.2.3
2026-08-21 05:42:31 +0200: Memory Usage (450040K) : Loading database to memory is complete.
2026-08-21 05:42:31 +0200: Memory Usage (450040K) : GF180MCU Klayout DRC runset output at: /foss/designs/layout/buffer_main.lyrdb
2026-08-21 05:42:31 +0200: Memory Usage (450040K) : Evaluate switches.
2026-08-21 05:42:31 +0200: Memory Usage (450040K) : table_name selected  main
2026-08-21 05:42:31 +0200: Memory Usage (450040K) : CONNECTIVITY_RULES enabled: true
2026-08-21 05:42:31 +0200: Memory Usage (450040K) : Wedge enabled: false
2026-08-21 05:42:31 +0200: Memory Usage (450040K) : Bal

In [75]:
verificar_drc_manual("buffer.gds")

?? Ejecutando DRC para: /foss/designs/layout/buffer.gds ...
?? DRC finalizó con errores de ejecución (Revisar logs).
?? Reporte de DRC detectado en: /foss/designs/layout/buffer_main.lyrdb
?? Limpiando memoria de KLayout...
?? Abriendo KLayout...
2026-08-21 06:55:23 +0200: Memory Usage (3107204K) : Starting running GF180MCU Klayout LVS runset on 
2026-08-21 06:55:23 +0200: Memory Usage (3107204K) : Ruby Version for klayout: 3.2.3
2026-08-21 06:55:23 +0200: Memory Usage (3107204K) : Loading database to memory is complete.
2026-08-21 06:55:23 +0200: Memory Usage (3107204K) : GF180MCU Klayout LVS runset output at default location: buffer.lvsdb
2026-08-21 06:55:23 +0200: Memory Usage (3107204K) : Evaluate switches.
2026-08-21 06:55:23 +0200: Memory Usage (3107204K) : Substrate name used: gf180mcu_gnd
2026-08-21 06:55:23 +0200: Memory Usage (3107204K) : Extracted netlist with net names: true
2026-08-21 06:55:23 +0200: Memory Usage (3107204K) : Extracted netlist with comments in details: fals

In [31]:
import time
from gdsfactory import Component
from glayout.flow.primitives.fet import nmos, pmos
from glayout.flow.primitives.guardring import tapring
from glayout.flow.routing.c_route import c_route
from glayout.flow.routing.straight_route import straight_route
from glayout.flow.routing.L_route import L_route
from glayout.flow.pdk.gf180_mapped import gf180_mapped_pdk as gf180

def create_folded_cascode_circuit(pdk_config):
    
    top_level = Component("folded_cascode_circuit")

    # ==========================================
    # 1. Configuración de Celdas (fingers=1)
    # ==========================================
    fet_kwargs_nmos = {
        "with_tie": False, "with_substrate_tap": False, "with_dummy": False,
        "with_dnwell": False, "sd_route_topmet": "met3", "gate_route_topmet": "met2"
    }
    fet_kwargs_pmos = {
        "with_tie": False, "with_substrate_tap": False, "with_dummy": False,
        "dnwell": False, "sd_route_topmet": "met3", "gate_route_topmet": "met2"
    }

    xm3  = top_level << pmos(pdk_config, width=32.0, length=1.0, fingers=1, **fet_kwargs_pmos)
    xm1  = top_level << pmos(pdk_config, width=10.0, length=1.0, fingers=1, **fet_kwargs_pmos)
    xm2  = top_level << pmos(pdk_config, width=10.0, length=1.0, fingers=1, **fet_kwargs_pmos)
    xm10 = top_level << pmos(pdk_config, width=4.0, length=1.0, fingers=1, **fet_kwargs_pmos)
    xm11 = top_level << pmos(pdk_config, width=4.0, length=1.0, fingers=1, **fet_kwargs_pmos)
    xm8  = top_level << pmos(pdk_config, width=4.0, length=1.0, fingers=1, **fet_kwargs_pmos)
    xm9  = top_level << pmos(pdk_config, width=4.0, length=1.0, fingers=1, **fet_kwargs_pmos)

    xm6  = top_level << nmos(pdk_config, width=4.0, length=1.0, fingers=1, **fet_kwargs_nmos)
    xm7  = top_level << nmos(pdk_config, width=4.0, length=1.0, fingers=1, **fet_kwargs_nmos)
    xm4  = top_level << nmos(pdk_config, width=9.0, length=1.0, fingers=1, **fet_kwargs_nmos)
    xm5  = top_level << nmos(pdk_config, width=9.0, length=1.0, fingers=1, **fet_kwargs_nmos)

    # ==========================================
    # 2. Floorplan Ultra-Compacto (Filas Secuenciales)
    # ==========================================
    x_inner = 3.5
    x_outer = 7.5
    y_gap = 1.5
    
    # Fila 1: Tail y Vbfb (Alineados en la parte superior para ahorrar espacio)
    xm3.x = 0.0
    xm3.y = 0.0
    xm10.x = -x_outer
    xm11.x = x_outer
    xm10.ymax = xm11.ymax = xm3.ymax
    
    # Fila 2: Par Diferencial (Justo debajo del Tail)
    xm1.x = -x_inner
    xm2.x = x_inner
    xm1.ymax = xm2.ymax = xm3.ymin - y_gap
    
    # Fila 3: Cascodes PMOS (Justo debajo del Par Diferencial)
    xm8.x = -x_outer
    xm9.x = x_outer
    xm8.ymax = xm9.ymax = xm1.ymin - y_gap
    
    # Fila 4: Cascodes NMOS (Con el gap necesario para NWELL a P-Sub)
    y_gap_p2n = 3.5
    xm6.x = -x_outer
    xm7.x = x_outer
    xm6.ymax = xm7.ymax = xm8.ymin - y_gap_p2n
    
    # Fila 5: Sinks NMOS
    xm4.x = -x_outer
    xm5.x = x_outer
    xm4.ymax = xm5.ymax = xm6.ymin - y_gap

    # ==========================================
    # 3. Anillos de Sustrato Ajustados (Tight Fit)
    # ==========================================
    pmos_xmin = min(xm10.xmin, xm8.xmin, xm1.xmin)
    pmos_xmax = max(xm11.xmax, xm9.xmax, xm2.xmax)
    pmos_ymin = xm8.ymin
    pmos_ymax = xm3.ymax
    
    ntap = top_level << tapring(pdk_config, enclosed_rectangle=((pmos_xmax - pmos_xmin) + 1.0, (pmos_ymax - pmos_ymin) + 1.0), sdlayer="n+s/d")
    ntap.x, ntap.y = (pmos_xmin + pmos_xmax) / 2.0, (pmos_ymin + pmos_ymax) / 2.0
    top_level.add_polygon([[ntap.xmin-0.5, ntap.ymin-0.5], [ntap.xmax+0.5, ntap.ymin-0.5], [ntap.xmax+0.5, ntap.ymax+0.5], [ntap.xmin-0.5, ntap.ymax+0.5]], layer=pdk_config.glayers["nwell"])

    nmos_xmin = min(xm6.xmin, xm4.xmin)
    nmos_xmax = max(xm7.xmax, xm5.xmax)
    nmos_ymin = xm4.ymin
    nmos_ymax = xm6.ymax
    
    ptap = top_level << tapring(pdk_config, enclosed_rectangle=((nmos_xmax - nmos_xmin) + 1.0, (nmos_ymax - nmos_ymin) + 1.0), sdlayer="p+s/d")
    ptap.x, ptap.y = (nmos_xmin + nmos_xmax) / 2.0, (nmos_ymin + nmos_ymax) / 2.0

    if "pwell" in pdk_config.glayers:
        top_level.add_polygon([[ptap.xmin, ptap.ymin], [ptap.xmax, ptap.ymin], [ptap.xmax, ptap.ymax], [ptap.xmin, ptap.ymax]], layer=pdk_config.glayers["pwell"])

    # ==========================================
    # 4. Enrutamiento Ortogonal Limpio
    # ==========================================
    # VDD y VSS
    top_level << c_route(pdk_config, xm10.ports['source_N'], xm3.ports['source_N'], extension=1.5)
    top_level << c_route(pdk_config, xm11.ports['source_N'], xm3.ports['source_N'], extension=1.5)
    top_level << straight_route(pdk_config, xm4.ports['source_E'], xm5.ports['source_W'], glayer1="met2") 

    # Conexiones de Cola (Tail)
    top_level << straight_route(pdk_config, xm1.ports['source_E'], xm2.ports['source_W'], glayer1="met2")
    top_level << L_route(pdk_config, xm3.ports['drain_S'], xm1.ports['source_E'], hglayer="met2", vglayer="met3")

    # Nodos Plegados (Folded)
    top_level << L_route(pdk_config, xm1.ports['drain_W'], xm6.ports['source_N'], hglayer="met2", vglayer="met3")
    top_level << L_route(pdk_config, xm2.ports['drain_E'], xm7.ports['source_N'], hglayer="met2", vglayer="met3")

    # Rutas Verticales (Cascodes y Salidas)
    top_level << straight_route(pdk_config, xm10.ports['drain_S'], xm8.ports['source_N'], glayer1="met3")
    top_level << straight_route(pdk_config, xm11.ports['drain_S'], xm9.ports['source_N'], glayer1="met3")
    top_level << straight_route(pdk_config, xm8.ports['drain_S'], xm6.ports['drain_N'], glayer1="met3")
    top_level << straight_route(pdk_config, xm9.ports['drain_S'], xm7.ports['drain_N'], glayer1="met3")
    top_level << straight_route(pdk_config, xm6.ports['source_S'], xm4.ports['drain_N'], glayer1="met3")
    top_level << straight_route(pdk_config, xm7.ports['source_S'], xm5.ports['drain_N'], glayer1="met3")

    # Buses de Compuerta (Cruzan de lado a lado por zonas 100% vacías)
    top_level << c_route(pdk_config, xm10.ports['gate_N'], xm11.ports['gate_N'], extension=1.5)
    top_level << straight_route(pdk_config, xm8.ports['gate_E'],  xm9.ports['gate_W'], glayer1="met2")  
    top_level << straight_route(pdk_config, xm6.ports['gate_E'],  xm7.ports['gate_W'], glayer1="met2")  
    top_level << straight_route(pdk_config, xm4.ports['gate_E'],  xm5.ports['gate_W'], glayer1="met2")  

    top_level = top_level.remove_layers(layers=[(204, 0)])
    return top_level

if __name__ == "__main__":
    folded_layout = create_folded_cascode_circuit(gf180)
    ruta_gds_folded = "folded.gds"
    folded_layout.write_gds(ruta_gds_folded)
    print("GDS Generado: Floorplan Secuencial Ultracompacto.")
    
    run_drc_v2(ruta_gds_folded)

/tmp/ipykernel_922/1361117107.py:133: UserWarning: Unnamed cells, 1 in 'Unnamed_c3182891'
  folded_layout.write_gds(ruta_gds_folded)
2026-08-22 22:14:09.583 | INFO     | gdsfactory.component:_write_library:1851 - Wrote to 'folded.gds'


GDS Generado: Floorplan Secuencial Ultracompacto.
?? Ejecutando DRC para: folded.gds ...
¿Diseño libre de errores (DRC Clean)?: False

----- DRC STDOUT -----
2026-08-22 22:14:12 +0200: Memory Usage (448616K) : Starting running GF180MCU Klayout DRC runset on /foss/designs/layout/folded.gds
2026-08-22 22:14:12 +0200: Memory Usage (448616K) : Ruby Version for klayout: 3.2.3
2026-08-22 22:14:12 +0200: Memory Usage (449620K) : Loading database to memory is complete.
2026-08-22 22:14:12 +0200: Memory Usage (449620K) : GF180MCU Klayout DRC runset output at: /foss/designs/layout/folded_main.lyrdb
2026-08-22 22:14:12 +0200: Memory Usage (449748K) : Evaluate switches.
2026-08-22 22:14:12 +0200: Memory Usage (449748K) : table_name selected  main
2026-08-22 22:14:12 +0200: Memory Usage (449748K) : CONNECTIVITY_RULES enabled: true
2026-08-22 22:14:12 +0200: Memory Usage (449748K) : Wedge enabled: false
2026-08-22 22:14:12 +0200: Memory Usage (449748K) : Ball enabled: false
2026-08-22 22:14:12 +0200

In [33]:
verificar_drc_manual("folded.gds")

?? Ejecutando DRC para: /foss/designs/layout/folded.gds ...
?? DRC finalizó con errores de ejecución (Revisar logs).
?? Reporte de DRC detectado en: /foss/designs/layout/folded_main.lyrdb
?? Limpiando memoria de KLayout...
?? Abriendo KLayout...
2026-08-23 02:00:42 +0200: Memory Usage (3107736K) : Starting running GF180MCU Klayout LVS runset on 
2026-08-23 02:00:42 +0200: Memory Usage (3107736K) : Ruby Version for klayout: 3.2.3
2026-08-23 02:00:42 +0200: Memory Usage (3107736K) : Loading database to memory is complete.
2026-08-23 02:00:42 +0200: Memory Usage (3107736K) : GF180MCU Klayout LVS runset output at default location: folded.lvsdb
2026-08-23 02:00:42 +0200: Memory Usage (3107736K) : Evaluate switches.
2026-08-23 02:00:42 +0200: Memory Usage (3107736K) : Substrate name used: gf180mcu_gnd
2026-08-23 02:00:42 +0200: Memory Usage (3107736K) : Extracted netlist with net names: true
2026-08-23 02:00:42 +0200: Memory Usage (3107736K) : Extracted netlist with comments in details: fals

In [41]:
verificar_drc_manual("folded_cascode_v3.gds")

?? Ejecutando DRC para: /foss/designs/layout/folded_cascode_v3.gds ...
?? DRC finalizó con errores de ejecución (Revisar logs).
?? Reporte de DRC detectado en: /foss/designs/layout/folded_cascode_v3_main.lyrdb
?? Limpiando memoria de KLayout...
?? Abriendo KLayout...
2026-08-23 05:01:06 +0200: Memory Usage (3113968K) : Starting running GF180MCU Klayout LVS runset on 
2026-08-23 05:01:06 +0200: Memory Usage (3113968K) : Ruby Version for klayout: 3.2.3
2026-08-23 05:01:06 +0200: Memory Usage (3113968K) : Loading database to memory is complete.
2026-08-23 05:01:06 +0200: Memory Usage (3113968K) : GF180MCU Klayout LVS runset output at default location: folded_cascode_v3.lvsdb
2026-08-23 05:01:06 +0200: Memory Usage (3113968K) : Evaluate switches.
2026-08-23 05:01:06 +0200: Memory Usage (3113968K) : Substrate name used: gf180mcu_gnd
2026-08-23 05:01:06 +0200: Memory Usage (3113968K) : Extracted netlist with net names: true
2026-08-23 05:01:06 +0200: Memory Usage (3113968K) : Extracted netli

In [8]:
verificar_drc_manual("switch_zero.gds")

?? Ejecutando DRC para: /foss/designs/layout/switch_zero.gds ...
?? DRC finalizó con errores de ejecución (Revisar logs).
?? Reporte de DRC detectado en: /foss/designs/layout/folded_cascode_v3_main.lyrdb
?? Limpiando memoria de KLayout...
?? Abriendo KLayout...


In [14]:
import math
from gdsfactory import Component
from glayout.flow.pdk.mappedpdk import MappedPDK
from glayout.flow.primitives.mimcap import mimcap
from glayout.flow.routing.straight_route import straight_route


def agregar_capa_marcado_lvs(comp: Component, pdk: MappedPDK, ref_inst):
    """Inyecta la capa de reconocimiento LVS (cap_mk) sobre cada celda MIM instanciada."""
    try:
        cap_mk_layer = pdk.get_glayer("cap_mk")
    except KeyError:
        # Capa por defecto de reconocimiento MIM cap en GF180MCU (Layer 117, datatype 5)
        cap_mk_layer = (117, 5)

    comp.add_polygon(
        [
            (ref_inst.xmin, ref_inst.ymin),
            (ref_inst.xmax, ref_inst.ymin),
            (ref_inst.xmax, ref_inst.ymax),
            (ref_inst.xmin, ref_inst.ymax),
        ],
        layer=cap_mk_layer,
    )


def agregar_etiqueta_lvs(
    comp: Component, pdk: MappedPDK, nombre_puerto: str, texto_etiqueta: str
):
    """Agrega labels de texto en la capa del puerto para que Netgen/KLayout identifique los nodos."""
    puerto = comp.ports[nombre_puerto]
    layer_pin = (
        pdk.get_glayer(puerto.layer)
        if isinstance(puerto.layer, str)
        else puerto.layer
    )
    comp.add_label(
        text=texto_etiqueta, position=puerto.center, layer=layer_pin
    )


def cap_mim_2f0fF(pdk: MappedPDK, width=10.0, length=10.0, multiplier=1):
    comp = Component(f"cap_mim_2f0fF_m{multiplier}")

    filas = int(math.isqrt(multiplier))
    while multiplier % filas != 0 and filas > 1:
        filas -= 1
    cols = math.ceil(multiplier / filas)

    separacion_x = 3.0
    separacion_y = 3.0

    grid = []
    contador = 0

    # 1. Instanciación e inyección del marcador LVS para todo el array
    for r in range(filas):
        fila_actual = []
        for c in range(cols):
            if contador >= multiplier:
                break

            # Genera la celda utilizando la PCell nativa de glayout
            mim_cell = mimcap(pdk, size=(width, length))
            mim_ref = comp << mim_cell

            mim_ref.x = c * (width + separacion_x)
            mim_ref.y = -r * (length + separacion_y)

            # Inyecta explícitamente el marcador 'cap_mk' en la posición exacta del capacitor
            agregar_capa_marcado_lvs(comp, pdk, mim_ref)

            fila_actual.append(mim_ref)
            contador += 1
        grid.append(fila_actual)

    # 2. Trazado Horizontal (Placas Superior e Inferior)
    for r in range(len(grid)):
        for c in range(len(grid[r]) - 1):
            comp << straight_route(
                pdk,
                grid[r][c].ports["top_met_E"],
                grid[r][c + 1].ports["top_met_W"],
            )
            comp << straight_route(
                pdk,
                grid[r][c].ports["bottom_met_E"],
                grid[r][c + 1].ports["bottom_met_W"],
            )

    # 3. Trazado Vertical
    for r in range(len(grid) - 1):
        comp << straight_route(
            pdk, grid[r][0].ports["top_met_S"], grid[r + 1][0].ports["top_met_N"]
        )
        comp << straight_route(
            pdk,
            grid[r][0].ports["bottom_met_S"],
            grid[r + 1][0].ports["bottom_met_N"],
        )

        col_fin_actual = len(grid[r]) - 1
        col_fin_sig = len(grid[r + 1]) - 1
        if col_fin_actual == col_fin_sig:
            comp << straight_route(
                pdk,
                grid[r][col_fin_actual].ports["top_met_S"],
                grid[r + 1][col_fin_sig].ports["top_met_N"],
            )
            comp << straight_route(
                pdk,
                grid[r][col_fin_actual].ports["bottom_met_S"],
                grid[r + 1][col_fin_sig].ports["bottom_met_N"],
            )

    # 4. Puertos Globales
    comp.add_port("TOP_W", port=grid[0][0].ports["top_met_W"])
    comp.add_port("BOT_W", port=grid[0][0].ports["bottom_met_W"])
    comp.add_port("TOP_E", port=grid[0][-1].ports["top_met_E"])
    comp.add_port("BOT_E", port=grid[0][-1].ports["bottom_met_E"])

    return comp


def pga_cf_bank(pdk: MappedPDK, width=10.0, length=10.0):
    bank = Component("pga_cf_bank")

    multiplicadores = [1, 2, 4, 8, 16]
    espaciado_bloques_x = 15.0
    separacion_ramas_y = 80.0

    pos_x_actual = 0.0

    for m in multiplicadores:
        # Rama Principal (P)
        cap_p = cap_mim_2f0fF(pdk, width=width, length=length, multiplier=m)
        ref_p = bank << cap_p
        ref_p.x = pos_x_actual
        ref_p.y = 0.0

        # Rama Complementaria (N)
        cap_n = cap_mim_2f0fF(pdk, width=width, length=length, multiplier=m)
        ref_n = bank << cap_n
        ref_n.x = pos_x_actual
        ref_n.y = -separacion_ramas_y

        # Puertos Generales
        bank.add_port(f"CfP_x{m}_TOP", port=ref_p.ports["TOP_W"])
        bank.add_port(f"CfP_x{m}_BOT", port=ref_p.ports["BOT_W"])
        bank.add_port(f"CfN_x{m}_TOP", port=ref_n.ports["TOP_W"])
        bank.add_port(f"CfN_x{m}_BOT", port=ref_n.ports["BOT_W"])

        # Etiquetas LVS en las terminales
        agregar_etiqueta_lvs(bank, pdk, f"CfP_x{m}_TOP", f"OUT_P_{m}")
        agregar_etiqueta_lvs(bank, pdk, f"CfP_x{m}_BOT", f"IN_P_{m}")
        agregar_etiqueta_lvs(bank, pdk, f"CfN_x{m}_TOP", f"OUT_N_{m}")
        agregar_etiqueta_lvs(bank, pdk, f"CfN_x{m}_BOT", f"IN_N_{m}")

        ancho_bloque = ref_p.xmax - ref_p.xmin
        pos_x_actual += ancho_bloque + espaciado_bloques_x

    return bank


# --- Generación del GDS Final ---
ruta_gds_caps = "/foss/designs/layout/capfb_m1.gds"
pga_layout = pga_cf_bank(gf180, width=10.0, length=10.0)
pga_layout.write_gds(ruta_gds_caps)
run_drc_v2(ruta_gds_caps)

/tmp/ipykernel_1016/723719710.py:168: UserWarning: Unnamed cells, 62 in 'pga_cf_bank$2'
  pga_layout.write_gds(ruta_gds_caps)
2026-08-23 18:02:07.414 | INFO     | gdsfactory.component:_write_library:1851 - Wrote to '/foss/designs/layout/capfb_m1.gds'


?? Ejecutando DRC para: /foss/designs/layout/capfb_m1.gds ...
¿Diseño libre de errores (DRC Clean)?: False

----- DRC STDOUT -----
2026-08-23 18:02:11 +0200: Memory Usage (447912K) : Starting running GF180MCU Klayout DRC runset on /foss/designs/layout/capfb_m1.gds
2026-08-23 18:02:11 +0200: Memory Usage (447912K) : Ruby Version for klayout: 3.2.3


----- DRC STDERR -----
/foss/pdks/ciel/gf180mcu/versions/7b70722e33c03fcb5dabcf4d479fb0822d9251c9/gf180mcuD/libs.tech/klayout/tech/drc/run_drc.py:729: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now_str = datetime.utcnow().strftime("drc_run_%Y_%m_%d_%H_%M_%S")
23-Aug-2026 18:02:08 | INFO    | Your Klayout version is: KLayout 0.30.8
23-Aug-2026 18:02:08 | INFO    | ## Generating template with for the following rule tables: ['dummy_metal4.drc', 'dummy_exclude.drc', 'nat_split.drc', 'du

In [28]:
import math
import numpy as np

# Parche de compatibilidad NumPy 2.0 para gdsfactory
if not hasattr(np, "float_"):
    np.float_ = np.float64

from gdsfactory import Component
from glayout.flow.pdk.gf180_mapped import gf180_mapped_pdk as gf180
from glayout.flow.pdk.mappedpdk import MappedPDK
from glayout.flow.primitives.mimcap import mimcap
from glayout.flow.routing.straight_route import straight_route


def pga_cf_bank(
    pdk: MappedPDK = gf180,
    width: float = 11.0,
    length: float = 11.0,
    multiplicadores: list[int] = [1, 2, 4, 8, 16],
    option: str = "B",
    with_extension: bool = True,
    extension_direction: str = "S",
    extension_width: float = 0.0,
    extension_length: float = 0.0,
    espaciado_bloques_x: float = 15.0,
    separacion_ramas_y: float = 80.0,
    separacion_grid: float = 4.0,
) -> Component:
    """Genera en una sola función el banco diferencial de capacitores PGA con Opción B (Met4-Met5)."""
    bank = Component("pga_cf_bank")
    pos_x_actual = 0.0

    for m in multiplicadores:
        # 1. Cálculo de filas y columnas para el arreglo 2D compacto
        filas = int(math.isqrt(m))
        while m % filas != 0 and filas > 1:
            filas -= 1
        cols = math.ceil(m / filas)

        grid_p = []
        grid_n = []
        contador = 0

        # 2. Instanciación en cuadrícula para Ramas P y N (Met4-Met5, Opción B)
        for r in range(filas):
            fila_p = []
            fila_n = []
            for c in range(cols):
                if contador >= m:
                    break

                # Celda MIM Opción B (Met4-Met5)
                mim_cell = mimcap(
                    pdk=pdk,
                    size=(width, length),
                    option=option,
                    with_extension=with_extension,
                    extension_direction=extension_direction,
                    extension_width=extension_width,
                    extension_length=extension_length,
                )

                # Colocación Rama Principal (P)
                ref_p = bank << mim_cell
                ref_p.x = pos_x_actual + c * (width + separacion_grid)
                ref_p.y = -r * (length + separacion_grid)
                fila_p.append(ref_p)

                # Colocación Rama Complementaria (N)
                ref_n = bank << mim_cell
                ref_n.x = pos_x_actual + c * (width + separacion_grid)
                ref_n.y = -separacion_ramas_y - r * (length + separacion_grid)
                fila_n.append(ref_n)

                contador += 1

            grid_p.append(fila_p)
            grid_n.append(fila_n)

        # 3. Enrutado interno en paralelo (Horizontal y Vertical)
        for grid in [grid_p, grid_n]:
            # Enrutado Horizontal (E -> W)
            for r in range(len(grid)):
                for c in range(len(grid[r]) - 1):
                    bank << straight_route(
                        pdk,
                        grid[r][c].ports["top_met_E"],
                        grid[r][c + 1].ports["top_met_W"],
                    )

            # Enrutado Vertical (S -> N)
            for r in range(len(grid) - 1):
                bank << straight_route(
                    pdk,
                    grid[r][0].ports["top_met_S"],
                    grid[r + 1][0].ports["top_met_N"],
                )

                c_fin_curr = len(grid[r]) - 1
                c_fin_next = len(grid[r + 1]) - 1
                if c_fin_curr == c_fin_next:
                    bank << straight_route(
                        pdk,
                        grid[r][c_fin_curr].ports["top_met_S"],
                        grid[r + 1][c_fin_next].ports["top_met_N"],
                    )

        # 4. Mapeo de Puertos Globales
        bank.add_port(f"CfP_x{m}_TOP_W", port=grid_p[0][0].ports["top_met_W"])
        bank.add_port(f"CfP_x{m}_TOP_E", port=grid_p[0][-1].ports["top_met_E"])
        bank.add_port(f"CfN_x{m}_TOP_W", port=grid_n[0][0].ports["top_met_W"])
        bank.add_port(f"CfN_x{m}_TOP_E", port=grid_n[0][-1].ports["top_met_E"])

        # Actualización de posición horizontal
        ancho_bloque = (
            cols * width + (cols - 1) * separacion_grid if cols > 0 else width
        )
        pos_x_actual += ancho_bloque + espaciado_bloques_x

    return bank


# --- Generación Directa de GDS ---
if __name__ == "__main__":
    ruta_gds = "/foss/designs/layout/pga_cf_bank.gds"

    # Parámetros tipo B (Met4-Met5) con extensión al Sur
    mimcap_kwargs = {
        "option": "B",
        "with_extension": True,
        "extension_direction": "S",
    }

    layout = pga_cf_bank(
        gf180, width=11.0, length=11.0, **mimcap_kwargs
    )
    layout.write_gds(ruta_gds)

    print(
        f"Banco PGA Opción B (Met4-Met5) generado exitosamente en: {ruta_gds}"
    )

    run_drc_v2(ruta_gds)

TypeError: mimcap() got an unexpected keyword argument 'option'

In [14]:
verificar_drc_manual(ruta_gds_cap_array)

?? Ejecutando DRC para: /foss/designs/layout/cap_mim_array.gds ...
?? DRC finalizó con errores de ejecución (Revisar logs).
?? Reporte de DRC detectado en: /foss/designs/layout/main.lyrdb
?? Limpiando memoria de KLayout...
?? Abriendo KLayout...
(22, 0)
## gf180mcu PDK Pcells loaded.
['/foss/pdks/ciel/gf180mcu/versions/7b70722e33c03fcb5dabcf4d479fb0822d9251c9/gf180mcuD/libs.tech/klayout/tech/pymacros', '/foss/tools/klayout/pymod', '/foss/tools/klayout_gdsfactory9/lib/python3.12/site-packages', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '/headless/.local/lib/python3.12/site-packages', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/headless/.klayout/python', '/headless/.klayout/salt/klive/python', '/headless/.klayout/salt/KLayoutPluginUtils/python', '/headless/.klayout/salt/gdsfactory/python']
klive 0.4.1 is running
